In [2]:
import cv2
import time
from ultralytics import YOLO
from IPython.display import Video, display

In [3]:
model = YOLO("yolov8n.pt")

In [4]:
def process_video(video_path, output_path, roi=None):
    
    # Open video
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Could not open video: {video_path}")
        return

    # Video information
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    original_fps = cap.get(cv2.CAP_PROP_FPS)

    # Output video
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(
        output_path,
        fourcc,
        original_fps,
        (width, height)
    )

    # Default ROI
    if roi is None:
        roi = (
            int(width * 0.25),
            int(height * 0.25),
            int(width * 0.75),
            int(height * 0.75)
        )

    x1, y1, x2, y2 = roi

    # Tracking information
    previous_inside = set()
    entered_ids = set()
    exited_ids = set()
    all_ids = set()

    # FPS calculation
    frame_count = 0
    start_time = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1

        # YOLO detection + tracking
        results = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False
        )

        current_inside = set()

        # Process detected objects
        if results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                # Tracking ID
                if box.id is None:
                    continue
                track_id = int(box.id[0])
                all_ids.add(track_id)

                # Bounding box
                x1_box, y1_box, x2_box, y2_box = map(
                    int, box.xyxy[0]
                )

                # Object center
                center_x = int((x1_box + x2_box) / 2)
                center_y = int((y1_box + y2_box) / 2)

                # Class name
                class_id = int(box.cls[0])
                class_name = model.names[class_id]

                # Check whether center is inside ROI
                inside = (
                    x1 <= center_x <= x2 and
                    y1 <= center_y <= y2
                )

                if inside:
                    current_inside.add(track_id)

                # Draw bounding box
                cv2.rectangle(
                    frame,
                    (x1_box, y1_box),
                    (x2_box, y2_box),
                    (0, 255, 0),
                    2
                )

                # Draw center
                cv2.circle(
                    frame,
                    (center_x, center_y),
                    5,
                    (0, 0, 255),
                    -1
                )

                # Display ID and class
                label = f"ID {track_id} - {class_name}"

                cv2.putText(
                    frame,
                    label,
                    (x1_box, y1_box - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )

        # Detect entries
        new_entries = current_inside - previous_inside

        for track_id in new_entries:
            entered_ids.add(track_id)

        # Detect exits
        new_exits = previous_inside - current_inside

        for track_id in new_exits:
            exited_ids.add(track_id)

        previous_inside = current_inside.copy()

        # Calculate processing FPS
        elapsed_time = time.time() - start_time
        processing_fps = frame_count / elapsed_time if elapsed_time > 0 else 0

        # Draw ROI
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            3
        )

        cv2.putText(
            frame,
            "ROI",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 0, 0),
            2
        )

        # Live statistics
        cv2.putText(
            frame,
            f"FPS: {processing_fps:.2f}",
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Current Objects: {len(current_inside)}",
            (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Total Unique Objects: {len(all_ids)}",
            (20, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Entries: {len(entered_ids)}",
            (20, 120),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Exits: {len(exited_ids)}",
            (20, 150),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        # Save frame
        out.write(frame)

    # Release resources
    cap.release()
    out.release()

    print("Video processing completed.")
    print(f"Total unique objects: {len(all_ids)}")
    print(f"Objects entered ROI: {len(entered_ids)}")
    print(f"Objects exited ROI: {len(exited_ids)}")
    print(f"Average processing FPS: {processing_fps:.2f}")

In [6]:
video1 = "videooo/traffic1.mp4"
output1 = "videos/output1.mp4"

process_video(
    video1,
    output1
)

Video processing completed.
Total unique objects: 36
Objects entered ROI: 19
Objects exited ROI: 18
Average processing FPS: 2.81


In [7]:
video2 = "videooo/traffic2.mp4"
output2 = "videos/output2.mp4"

process_video(
    video2,
    output2
)

Video processing completed.
Total unique objects: 144
Objects entered ROI: 72
Objects exited ROI: 68
Average processing FPS: 4.92


In [8]:
video3 = "videooo/traffic3.mp4"
output3 = "videos/output3.mp4"

process_video(
    video3,
    output3
)

Video processing completed.
Total unique objects: 60
Objects entered ROI: 20
Objects exited ROI: 18
Average processing FPS: 5.06
